# Supabase MCP Metadata Extraction Test

This notebook tests Supabase MCP by connecting to an MCP server, extracting schema metadata, and storing it in a local SQLite table.

Notes:
- For hosted Supabase MCP, OAuth login is required. You will be prompted to visit a URL and paste the callback URL.
- For local Supabase CLI MCP, OAuth is not required and the server URL is typically `http://localhost:54321/mcp`.


In [1]:
# If needed, install the MCP Python SDK
# %pip install "mcp[cli]"


In [2]:
import json
import os
import sqlite3
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv

# Load .env for notebook execution
env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(env_path)
else:
    alt_env = Path("saas-nl2sql-analytics-app/backend/.env")
    if alt_env.exists():
        load_dotenv(alt_env)

import httpx
from pydantic import AnyUrl
from mcp import ClientSession, types
from mcp.client.auth import OAuthClientProvider, OAuthTokenError, TokenStorage
from mcp.client.auth.utils import handle_token_response_scopes
from mcp.client.streamable_http import streamable_http_client
from mcp.shared.auth import OAuthClientInformationFull, OAuthClientMetadata, OAuthToken

# === MCP configuration ===
MCP_SERVER_URL = os.getenv(
    "SUPABASE_MCP_URL",
    "https://mcp.supabase.com/mcp?project_ref=usrqkxgtpjqsbmealtna&read_only=true&features=database",
)
USE_OAUTH = os.getenv("SUPABASE_MCP_USE_OAUTH", "true").lower() in ("1", "true", "yes")
MCP_PROJECT_ID = os.getenv("SUPABASE_PROJECT_ID")
MCP_SCHEMAS = [s.strip() for s in os.getenv("MCP_SCHEMAS", "public").split(",") if s.strip()]

SUPABASE_OAUTH_CLIENT_ID = os.getenv("SUPABASE_OAUTH_CLIENT_ID")
SUPABASE_OAUTH_CLIENT_SECRET = os.getenv("SUPABASE_OAUTH_CLIENT_SECRET")
SUPABASE_OAUTH_CLIENT_AUTH_METHOD = os.getenv("SUPABASE_OAUTH_CLIENT_AUTH_METHOD", "client_secret_post")
SUPABASE_OAUTH_SCOPE = os.getenv("SUPABASE_OAUTH_SCOPE", "analytics:read database:read")
OAUTH_REDIRECT_URI = AnyUrl("http://localhost:3000/callback")

print("MCP_SERVER_URL=", MCP_SERVER_URL)
print("USE_OAUTH=", USE_OAUTH)
print("MCP_SCHEMAS=", MCP_SCHEMAS)


MCP_SERVER_URL= https://mcp.supabase.com/mcp?project_ref=usrqkxgtpjqsbmealtna&read_only=true&features=database
USE_OAUTH= True
MCP_SCHEMAS= ['public']


In [3]:
# === JSON metadata contract (v1) ===
METADATA_CONTRACT_V1 = {
    "version": "v1",
    "generated_at": "ISO-8601 timestamp",
    "source": {
        "mcp_server_url": "...",
        "project_ref": "...",
        "schemas": ["public"],
    },
    "schemas": [
        {
            "schema": "public",
            "tables": [
                {
                    "name": "table_name",
                    "comment": "optional comment",
                    "columns": [
                        {
                            "name": "column_name",
                            "data_type": "text",
                            "is_nullable": False,
                            "default": None,
                            "ordinal_position": 1,
                            "comment": "optional comment",
                        }
                    ],
                    "primary_key": ["id"],
                    "unique_constraints": [
                        {"name": "constraint_name", "columns": ["col"]}
                    ],
                    "foreign_keys": [
                        {
                            "name": "fk_name",
                            "columns": ["col"],
                            "references": {
                                "schema": "public",
                                "table": "other_table",
                                "columns": ["id"],
                            },
                        }
                    ],
                }
            ],
        }
    ],
    "relationships": [
        {
            "from": {"schema": "public", "table": "child", "columns": ["parent_id"]},
            "to": {"schema": "public", "table": "parent", "columns": ["id"]},
            "constraint": "child_parent_fkey",
        }
    ],
}

print(json.dumps(METADATA_CONTRACT_V1, indent=2))


{
  "version": "v1",
  "generated_at": "ISO-8601 timestamp",
  "source": {
    "mcp_server_url": "...",
    "project_ref": "...",
    "schemas": [
      "public"
    ]
  },
  "schemas": [
    {
      "schema": "public",
      "tables": [
        {
          "name": "table_name",
          "comment": "optional comment",
          "columns": [
            {
              "name": "column_name",
              "data_type": "text",
              "is_nullable": false,
              "default": null,
              "ordinal_position": 1,
              "comment": "optional comment"
            }
          ],
          "primary_key": [
            "id"
          ],
          "unique_constraints": [
            {
              "name": "constraint_name",
              "columns": [
                "col"
              ]
            }
          ],
          "foreign_keys": [
            {
              "name": "fk_name",
              "columns": [
                "col"
              ],
              "r

In [4]:
# === SQLite storage for metadata ===
DB_PATH = Path("metadata_test.sqlite3")

def init_metadata_db() -> None:
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("pragma journal_mode = wal")
        conn.execute(
            """
            create table if not exists db_metadata (
                id integer primary key autoincrement,
                source text not null,
                metadata_json text not null,
                created_at text not null
            )
            """
        )
        conn.commit()

def store_metadata(source: str, metadata: Dict[str, Any]) -> int:
    payload = json.dumps(metadata)
    created_at = datetime.now(timezone.utc).isoformat()
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.execute(
            "insert into db_metadata (source, metadata_json, created_at) values (?, ?, ?)",
            (source, payload, created_at),
        )
        conn.commit()
        return int(cur.lastrowid)

init_metadata_db()
print("DB ready at", DB_PATH)


DB ready at metadata_test.sqlite3


In [5]:
# === MCP helpers ===


class SupabaseOAuthClientProvider(OAuthClientProvider):
    """Accept Supabase token responses with HTTP 201 status."""

    async def _handle_token_response(self, response: httpx.Response) -> None:
        if response.status_code not in (200, 201):
            body = await response.aread()
            body_text = body.decode("utf-8")
            raise OAuthTokenError(f"Token exchange failed ({response.status_code}): {body_text}")

        token_response = await handle_token_response_scopes(response)
        self.context.current_tokens = token_response
        self.context.update_token_expiry(token_response)
        await self.context.storage.set_tokens(token_response)


class InMemoryTokenStorage(TokenStorage):
    """Simple in-memory token storage for MCP OAuth."""

    def __init__(self) -> None:
        self.tokens: Optional[OAuthToken] = None
        self.client_info: Optional[OAuthClientInformationFull] = None
        if SUPABASE_OAUTH_CLIENT_ID and SUPABASE_OAUTH_CLIENT_SECRET:
            self.client_info = OAuthClientInformationFull(
                client_id=SUPABASE_OAUTH_CLIENT_ID,
                client_secret=SUPABASE_OAUTH_CLIENT_SECRET,
                token_endpoint_auth_method=SUPABASE_OAUTH_CLIENT_AUTH_METHOD,
                redirect_uris=[OAUTH_REDIRECT_URI],
                grant_types=["authorization_code", "refresh_token"],
                response_types=["code"],
                scope=SUPABASE_OAUTH_SCOPE,
                client_name="Supabase MCP Notebook Client",
            )

    async def get_tokens(self) -> Optional[OAuthToken]:
        return self.tokens

    async def set_tokens(self, tokens: OAuthToken) -> None:
        self.tokens = tokens

    async def get_client_info(self) -> Optional[OAuthClientInformationFull]:
        return self.client_info

    async def set_client_info(self, client_info: OAuthClientInformationFull) -> None:
        if client_info is not None and client_info.client_secret and not client_info.token_endpoint_auth_method:
            client_info.token_endpoint_auth_method = "client_secret_post"
        self.client_info = client_info


OAUTH_EXPECTED_STATE: Optional[str] = None


async def handle_redirect(auth_url: str) -> None:
    global OAUTH_EXPECTED_STATE
    print("Visit this URL to authorize:")
    print(auth_url)
    try:
        from urllib.parse import parse_qs, urlparse

        params = parse_qs(urlparse(auth_url).query)
        OAUTH_EXPECTED_STATE = params.get("state", [None])[0]
    except Exception:
        OAUTH_EXPECTED_STATE = None


async def handle_callback() -> Tuple[str, Optional[str]]:
    raw = input("Paste callback URL (or just the code): ").strip()
    from urllib.parse import parse_qs, urlparse

    if raw.startswith("http://") or raw.startswith("https://"):
        params = parse_qs(urlparse(raw).query)
        return params["code"][0], params.get("state", [None])[0]

    if "code=" in raw:
        params = parse_qs(raw.lstrip("?"))
        return params["code"][0], params.get("state", [None])[0]

    return raw, OAUTH_EXPECTED_STATE


async def get_http_client() -> httpx.AsyncClient:
    if not USE_OAUTH:
        return httpx.AsyncClient(follow_redirects=True)

    oauth_auth = SupabaseOAuthClientProvider(
        server_url=MCP_SERVER_URL,
        client_metadata=OAuthClientMetadata(
            client_name="Supabase MCP Notebook Client",
            redirect_uris=[OAUTH_REDIRECT_URI],
            grant_types=["authorization_code", "refresh_token"],
            response_types=["code"],
            token_endpoint_auth_method=SUPABASE_OAUTH_CLIENT_AUTH_METHOD,
            scope=SUPABASE_OAUTH_SCOPE,
        ),
        storage=InMemoryTokenStorage(),
        redirect_handler=handle_redirect,
        callback_handler=handle_callback,
    )
    return httpx.AsyncClient(auth=oauth_auth, follow_redirects=True)


def _first_json_in_text(text: str) -> Optional[Any]:
    """Try to find and parse the first JSON object/array in a text blob."""
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch not in ("{", "["):
            continue
        try:
            obj, _ = decoder.raw_decode(text[i:])
            return obj
        except json.JSONDecodeError:
            continue
    return None


def _parse_tool_result(result: Any) -> Any:
    if getattr(result, "isError", False):
        messages = []
        for content in result.content:
            if isinstance(content, types.TextContent):
                messages.append(content.text)
        raise RuntimeError("Tool call failed: " + " | ".join(messages))

    structured = getattr(result, "structuredContent", None)
    if structured:
        # Some servers wrap JSON in a string field (e.g., "result")
        if isinstance(structured, dict):
            for value in structured.values():
                if isinstance(value, str):
                    data = _first_json_in_text(value)
                    if data is not None:
                        return data
        return structured

    text_parts = [c.text for c in result.content if isinstance(c, types.TextContent)]
    for text in text_parts:
        data = _first_json_in_text(text)
        if data is not None:
            return data

    return {"raw": text_parts}


def _normalize_rows(payload: Any) -> List[Dict[str, Any]]:
    if isinstance(payload, list):
        return payload
    if isinstance(payload, str):
        data = _first_json_in_text(payload)
        if isinstance(data, list):
            return data
    if isinstance(payload, dict):
        for key in ("data", "rows", "result", "records"):
            value = payload.get(key)
            if isinstance(value, list):
                return value
            if isinstance(value, str):
                data = _first_json_in_text(value)
                if isinstance(data, list):
                    return data
    return []


def _resolve_sql_key(input_schema: Dict[str, Any]) -> str:
    props = (input_schema or {}).get("properties", {}) or {}
    for key in ("sql", "query", "statement"):
        if key in props:
            return key
    required = (input_schema or {}).get("required", []) or []
    if len(required) == 1:
        return required[0]
    raise ValueError("Unable to infer SQL argument name for execute_sql tool.")


def _inject_required_args(input_schema: Dict[str, Any], args: Dict[str, Any]) -> Dict[str, Any]:
    required = (input_schema or {}).get("required", []) or []
    props = (input_schema or {}).get("properties", {}) or {}
    filled = dict(args)
    if "project_id" in required and "project_id" not in filled:
        if not MCP_PROJECT_ID:
            raise ValueError("project_id is required by this MCP tool. Set SUPABASE_PROJECT_ID.")
        filled["project_id"] = MCP_PROJECT_ID
    if "schemas" in props and "schemas" not in filled and MCP_SCHEMAS:
        filled["schemas"] = MCP_SCHEMAS
    return filled


In [6]:
# === Metadata extraction ===
async def _execute_sql(session: ClientSession, sql: str) -> List[Dict[str, Any]]:
    tools = await session.list_tools()
    tool_map = {tool.name: tool for tool in tools.tools}
    if "execute_sql" not in tool_map:
        raise RuntimeError("execute_sql tool is not available. Ensure MCP server has database feature enabled.")
    input_schema = tool_map["execute_sql"].inputSchema or {}
    sql_key = _resolve_sql_key(input_schema)
    args = _inject_required_args(input_schema, {sql_key: sql})
    result = await session.call_tool("execute_sql", arguments=args)
    payload = _parse_tool_result(result)
    return _normalize_rows(payload)


def _build_metadata(
    tables: List[Dict[str, Any]],
    columns: List[Dict[str, Any]],
    constraints: List[Dict[str, Any]],
    table_comments: List[Dict[str, Any]],
    column_comments: List[Dict[str, Any]],
) -> Dict[str, Any]:
    table_comment_map = {
        (row["table_schema"], row["table_name"]): row.get("table_comment")
        for row in table_comments
        if row.get("table_schema") and row.get("table_name")
    }
    column_comment_map = {
        (row["table_schema"], row["table_name"], row["column_name"]): row.get("column_comment")
        for row in column_comments
        if row.get("table_schema") and row.get("table_name") and row.get("column_name")
    }

    schema_map: Dict[str, Dict[str, Any]] = {}
    table_map: Dict[Tuple[str, str], Dict[str, Any]] = {}

    for row in tables:
        schema = row.get("table_schema")
        table = row.get("table_name")
        if not schema or not table:
            continue
        schema_entry = schema_map.setdefault(schema, {"schema": schema, "tables": []})
        table_entry = {
            "name": table,
            "comment": table_comment_map.get((schema, table)),
            "columns": [],
            "primary_key": [],
            "unique_constraints": [],
            "foreign_keys": [],
        }
        schema_entry["tables"].append(table_entry)
        table_map[(schema, table)] = table_entry

    for row in columns:
        schema = row.get("table_schema")
        table = row.get("table_name")
        if not schema or not table:
            continue
        table_entry = table_map.setdefault(
            (schema, table),
            {
                "name": table,
                "comment": table_comment_map.get((schema, table)),
                "columns": [],
                "primary_key": [],
                "unique_constraints": [],
                "foreign_keys": [],
            },
        )
        if schema not in schema_map:
            schema_map[schema] = {"schema": schema, "tables": [table_entry]}

        table_entry["columns"].append(
            {
                "name": row.get("column_name"),
                "data_type": row.get("data_type"),
                "is_nullable": (row.get("is_nullable") == "YES"),
                "default": row.get("column_default"),
                "ordinal_position": row.get("ordinal_position"),
                "comment": column_comment_map.get((schema, table, row.get("column_name"))),
            }
        )

    constraint_groups: Dict[Tuple[str, str, str, str], Dict[str, Any]] = defaultdict(
        lambda: {"columns": [], "ref_schema": None, "ref_table": None, "ref_columns": []}
    )

    for row in constraints:
        schema = row.get("table_schema")
        table = row.get("table_name")
        name = row.get("constraint_name")
        ctype = row.get("constraint_type")
        if not schema or not table or not name or not ctype:
            continue
        key = (schema, table, name, ctype)
        group = constraint_groups[key]
        if row.get("column_name"):
            group["columns"].append((row.get("ordinal_position") or 0, row.get("column_name")))
        if ctype == "FOREIGN KEY":
            group["ref_schema"] = row.get("foreign_table_schema")
            group["ref_table"] = row.get("foreign_table_name")
            if row.get("foreign_column_name"):
                group["ref_columns"].append((row.get("ordinal_position") or 0, row.get("foreign_column_name")))

    relationships: List[Dict[str, Any]] = []
    for (schema, table, name, ctype), group in constraint_groups.items():
        columns = [col for _, col in sorted(group["columns"], key=lambda x: x[0])]
        table_entry = table_map.get((schema, table))
        if not table_entry:
            continue
        if ctype == "PRIMARY KEY":
            table_entry["primary_key"] = columns
        elif ctype == "UNIQUE":
            table_entry["unique_constraints"].append({"name": name, "columns": columns})
        elif ctype == "FOREIGN KEY":
            ref_columns = [col for _, col in sorted(group["ref_columns"], key=lambda x: x[0])]
            fk = {
                "name": name,
                "columns": columns,
                "references": {
                    "schema": group["ref_schema"],
                    "table": group["ref_table"],
                    "columns": ref_columns,
                },
            }
            table_entry["foreign_keys"].append(fk)
            relationships.append(
                {
                    "from": {"schema": schema, "table": table, "columns": columns},
                    "to": {"schema": group["ref_schema"], "table": group["ref_table"], "columns": ref_columns},
                    "constraint": name,
                }
            )

    return {
        "version": "v1",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "source": {
            "mcp_server_url": MCP_SERVER_URL,
            "project_ref": MCP_PROJECT_ID or "(scoped via MCP URL)",
            "schemas": MCP_SCHEMAS,
        },
        "schemas": list(schema_map.values()),
        "relationships": relationships,
    }


async def extract_metadata() -> Dict[str, Any]:
    async with (await get_http_client()) as http_client:
        async with streamable_http_client(MCP_SERVER_URL, http_client=http_client) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                schema_list = ", ".join([f"'{s}'" for s in MCP_SCHEMAS]) if MCP_SCHEMAS else "'public'"
                schema_filter = f"and table_schema in ({schema_list})"
                schema_filter_tc = f"and tc.table_schema in ({schema_list})"
                schema_filter_nsp = f"and n.nspname in ({schema_list})"

                tables_sql = f"""
                    select table_schema, table_name
                    from information_schema.tables
                    where table_type = 'BASE TABLE'
                      and table_schema not in ('pg_catalog', 'information_schema')
                      {schema_filter}
                    order by table_schema, table_name;
                """
                columns_sql = f"""
                    select table_schema, table_name, column_name, data_type, is_nullable, column_default, ordinal_position
                    from information_schema.columns
                    where table_schema not in ('pg_catalog', 'information_schema')
                      {schema_filter}
                    order by table_schema, table_name, ordinal_position;
                """
                constraints_sql = f"""
                    select
                        tc.table_schema,
                        tc.table_name,
                        tc.constraint_name,
                        tc.constraint_type,
                        kcu.column_name,
                        kcu.ordinal_position,
                        ccu.table_schema as foreign_table_schema,
                        ccu.table_name as foreign_table_name,
                        ccu.column_name as foreign_column_name
                    from information_schema.table_constraints tc
                    left join information_schema.key_column_usage kcu
                        on tc.constraint_name = kcu.constraint_name
                        and tc.table_schema = kcu.table_schema
                    left join information_schema.constraint_column_usage ccu
                        on tc.constraint_name = ccu.constraint_name
                        and tc.table_schema = ccu.table_schema
                    where tc.table_schema not in ('pg_catalog', 'information_schema')
                      {schema_filter_tc};
                """
                table_comments_sql = f"""
                    select n.nspname as table_schema, c.relname as table_name, d.description as table_comment
                    from pg_catalog.pg_class c
                    join pg_catalog.pg_namespace n on n.oid = c.relnamespace
                    left join pg_catalog.pg_description d on d.objoid = c.oid and d.objsubid = 0
                    where c.relkind = 'r'
                      and n.nspname not in ('pg_catalog', 'information_schema')
                      {schema_filter_nsp};
                """
                column_comments_sql = f"""
                    select
                        n.nspname as table_schema,
                        c.relname as table_name,
                        a.attname as column_name,
                        d.description as column_comment
                    from pg_catalog.pg_class c
                    join pg_catalog.pg_namespace n on n.oid = c.relnamespace
                    join pg_catalog.pg_attribute a
                        on a.attrelid = c.oid
                        and a.attnum > 0
                        and not a.attisdropped
                    left join pg_catalog.pg_description d
                        on d.objoid = c.oid and d.objsubid = a.attnum
                    where c.relkind = 'r'
                      and n.nspname not in ('pg_catalog', 'information_schema')
                      {schema_filter_nsp};
                """

                tables_rows = await _execute_sql(session, tables_sql)
                columns_rows = await _execute_sql(session, columns_sql)
                constraints_rows = await _execute_sql(session, constraints_sql)
                table_comments_rows = await _execute_sql(session, table_comments_sql)
                column_comments_rows = await _execute_sql(session, column_comments_sql)

                return _build_metadata(
                    tables_rows,
                    columns_rows,
                    constraints_rows,
                    table_comments_rows,
                    column_comments_rows,
                )


In [8]:
# === Run extraction (Jupyter supports top-level await) ===
metadata = await extract_metadata()
print("Tables extracted:", sum(len(s["tables"]) for s in metadata["schemas"]))
row_id = store_metadata("supabase_mcp", metadata)
print("Stored metadata row id:", row_id)

# Preview
print(json.dumps(metadata, indent=2)[:2000])


Visit this URL to authorize:
https://api.supabase.com/v1/oauth/authorize?response_type=code&client_id=709f7338-d717-4dd3-9e95-68bceba3b90b&redirect_uri=http%3A%2F%2Flocalhost%3A3000%2Fcallback&state=CYhnv5aOj2k7gvf6Bqhaa7WLWlMnwoIvLRcrekAI9ME&code_challenge=jhVj5lCa8BgWi7NvoXp-8nv10AuyJu1zoDeUutAs2QI&code_challenge_method=S256&resource=https%3A%2F%2Fmcp.supabase.com%2Fmcp%3Fproject_ref%3Dusrqkxgtpjqsbmealtna%26read_only%3Dtrue%26features%3Ddatabase&scope=organizations%3Aread+projects%3Aread+projects%3Awrite+database%3Awrite+database%3Aread+analytics%3Aread+secrets%3Aread+edge_functions%3Aread+edge_functions%3Awrite+environment%3Aread+environment%3Awrite+storage%3Aread


Session termination failed: 404


Tables extracted: 1
Stored metadata row id: 12
{
  "version": "v1",
  "generated_at": "2026-03-14T23:33:01.053372+00:00",
  "source": {
    "mcp_server_url": "https://mcp.supabase.com/mcp?project_ref=usrqkxgtpjqsbmealtna&read_only=true&features=database",
    "project_ref": "usrqkxgtpjqsbmealtna",
    "schemas": [
      "public"
    ]
  },
  "schemas": [
    {
      "schema": "public",
      "tables": [
        {
          "name": "finance_economics_dataset",
          "comment": null,
          "columns": [
            {
              "name": "Date",
              "data_type": "text",
              "is_nullable": true,
              "default": null,
              "ordinal_position": 1,
              "comment": null
            },
            {
              "name": "Stock Index",
              "data_type": "text",
              "is_nullable": true,
              "default": null,
              "ordinal_position": 2,
              "comment": null
            },
            {
           

In [3]:
import sqlite3
from pathlib import Path

# Load and print rows from the SQLite DB
_db_path = (Path.cwd() / "mcp.sqlite3").resolve()
conn = sqlite3.connect(_db_path)
cur = conn.cursor()

# List tables
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = [row[0] for row in cur.fetchall()]
print("Tables:", tables)

# Print up to 20 rows per table
for t in tables:
    print(f"\n== {t} ==")
    cur.execute(f"SELECT * FROM {t} LIMIT 20;")
    rows = cur.fetchall()
    for r in rows:
        print(r)

conn.close()


Tables: ['db_metadata', 'mcp_auth_states', 'mcp_tokens', 'oauth_clients']

== db_metadata ==
('user_3Am5FISHis6y1Gjk0enyyzMtC5l', '{"version": "v1", "generated_at": "2026-03-18T21:14:17.073382+00:00", "source": {"mcp_server_url": "https://mcp.supabase.com/mcp?project_ref=usrqkxgtpjqsbmealtna&read_only=true&features=database", "project_ref": "usrqkxgtpjqsbmealtna"}, "schemas": [{"schema": "public", "tables": [{"name": "branch_finance_kpis", "comment": null, "columns": [{"name": "kpi_id", "data_type": "bigint", "is_nullable": false, "default": "nextval(\'branch_finance_kpis_kpi_id_seq\'::regclass)", "ordinal_position": 1, "comment": null}, {"name": "report_month", "data_type": "date", "is_nullable": false, "default": null, "ordinal_position": 2, "comment": null}, {"name": "branch_name", "data_type": "text", "is_nullable": false, "default": null, "ordinal_position": 3, "comment": null}, {"name": "region", "data_type": "text", "is_nullable": false, "default": null, "ordinal_position": 4, "